<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/An%20Advanced%20Heuristic%20Agent%20for%20Kaggriculture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
"""
main.py -- An advanced heuristic agent for Kaggriculture.
STRATEGY OVERVIEW:
1. Bootstrap on Wheat: Cheap ($10), fast payback (4 days). Builds initial cash.
2. Diversify: Add Carrots and Melons once cash allows to avoid market crashes.
3. Passive Income: Build coops/pastures and buy animals. They produce indefinitely.
4. Scale: Hire farm hands and buy land quadrants when the farm gets crowded.
5. Job-Queue System: Every turn, we scan the farm, create a prioritized list of
   "jobs" (Feed > Water > Harvest > Plant), and assign workers to the nearest job.
"""

import math

In [14]:
#1.STATIC GAME DATA (tHE "rULEBOOK")
WHEAT,CARROT,TOMATO,STRAWBERRY,MELON = "WHEAT","CARROT","TOMATO","STRAWBERRY","MELON"
GOOSE,COW,SHEEP = "GOOSE","COW","SHEEP"
FERTILIZER ="FERTILIZER"

ONE_TIME_CROPS = {WHEAT,CARROT,MELON}
ONGOING_CROPS = {TOMATO,STRAWBERRY}
ALL_CROPS = ONE_TIME_CROPS | ONGOING_CROPS

#Crop stats:seed cost,base sell price,day it hits max yield

CROP_INFO={
    WHEAT: dict(seed_cost=10,price=25,max_yield_day=4),
    CARROT: dict(seed_cost=20,price=35,max_yield_day=3),
    MELON: dict(seed_cost=80,price=250,max_yield_day=10),
    TOMATO: dict(seed_cost=50,price=60,max_yield_day=11),
    STRAWBERRY: dict(seed_cost=100,price=120,max_yield_day=16),
}
ANIMAL_INFO={
    GOOSE:dict(cost=300,product="EGG",structure="COOP",max_held=4),
    COW: dict(cost=400,product="MILK",structure="PASTURE",max_held=6),
    SHEEP:dict(cost=500,product="WOOL",structure="PASTURE",max_held=6),
}
SELLABLE_PRODUCTS = [WHEAT,CARROT,TOMATO,STRAWBERRY,MELON,"EGG","MILK","WOOL"]


In [15]:
def get_most_cost_effective_crop():
    """Determines the most cost-effective crop to plant based on price per seed cost,
    considering only crops in PLANT_PRIORITY."""
    most_effective_crop = None
    max_effectiveness = -1

    for crop_name in PLANT_PRIORITY:
        if crop_name in CROP_INFO:
            info = CROP_INFO[crop_name]
            seed_cost = info['seed_cost']
            price = info['price']

            if seed_cost > 0:
                effectiveness = price / seed_cost
                if effectiveness > max_effectiveness:
                    max_effectiveness = effectiveness
                    most_effective_crop = crop_name
    return most_effective_crop

# Demonstrate the helper function
cost_effective_crop = get_most_cost_effective_crop()
print(f"The most cost-effective crop to plant is: {cost_effective_crop}")

The most cost-effective crop to plant is: MELON


In [16]:
#2.CONFIGURATION
WHEAT_RESERVE_PER_ANIMAL =3
CASH_BUFFER = 250
MAX_SEEDS_TO_HOLD={WHEAT:6,CARROT:3,MELON:2,TOMATO:2,STRAWBERRY:1}
PLANT_PRIORITY = [WHEAT,CARROT,MELON,TOMATO,]#pREFERENCE ORDER FOR PLANTING
HIRE_MONEY_FLOOR=600
LAND_BUY_FLOOR=800
LAND_COSTS = {"NE":1000,"SW":2000,"SE":4000}
QUADRANT_ORDER=["NE","SW","SE"]
ANIMAL_BUILD_MONEY={GOOSE:700,COW:1200,SHEEP:1500}#MIN MONEY BEFORE INVESTING IN EAC
TARGET_ANIMALS={GOOSE:2,COW:1,SHEEP:1} #Eventual target counts of each animal

In [17]:
#3.HELPER FUNCTIONS
def manhattan(a,b):
  "Calculates grid distance between two points"
  return abs(a[0]-b[0])+abs(a[1]-b[1])

def step_toward(pos, target):
  "Returns the direction string (e.g., 'NORTH', 'SOUTHEAST') to move 1 step closer to target, including diagonal movement."
  x, y = pos
  tx, ty = target
  dx, dy = tx - x, ty - y

  if dx == 0 and dy == 0:
    return None

  direction = ""
  if dy < 0: # Target is north of current position
    direction += "NORTH"
  elif dy > 0: # Target is south of current position
    direction += "SOUTH"

  if dx > 0: # Target is east of current position
    direction += "EAST"
  elif dx < 0: # Target is west of current position
    direction += "WEST"

  return direction

def is_locked(farm,x,y):
  return farm["tiles"][y][x] =="LOCKED"

def is_animal_on_tile(farm, x, y):
    """Checks if an animal is on the given tile (x, y)."""
    for animal in farm['animals']:
        if animal['x'] == x and animal['y'] == y:
            return True
    return False


In [18]:
#4.JOB DISCOVERY(The "Brain":What needs to be done)
def find_jobs(farm,day,shed_adjacent):
  "Scans the farm and returns an ordered list of jobs TIERS(most urgent first)."
  feed_jobs,water_jobs,harvest_jobs = [],[],[]
  care_jobs,collect_fert_jobs,fertilize_jobs=[],[],[]
  place_jobs,plant_jobs,weed_jobs=[],[],[]

  tiles=farm["tiles"]
  for y,row in enumerate(tiles):
    for x,t in enumerate(row):
      if t is None: # Empty tile: can plant
        plant_jobs.append({"pos":(x,y),"action":None,"needs":"SEED","kind":"weed"})
      elif t == "LOCKED": # Locked tile: skip
        continue
      else: # 't' is a dictionary representing a tile with content
        # Check if it's a plant tile
        if "crop" in t:
          crop = t["crop"]
          age= day-t["planted_day"]

          #Priority 1: Water if not watered today
          if not t.get("watered_today",False):
            water_jobs.append({"pos":(x,y),"action":["WATER"],"needs":None,"kind":"water"})

          #Priority 2:Harvest if ready
          if t.get("yield_units",0)>0:
            if crop in ONGOING_CROPS:
              harvest_jobs.append({"pos":(x,y),"action":["HARVEST"],"needs":None,"kind":"harvest"})
            else:
              my_day=CROP_INFO[crop]["max_yield_day"]
              if age>=my_day:
                harvest_jobs.append({"pos":(x,y),"action":["HARVEST"],"needs":None,"kind":"harvest"})

          #Priority 3:Fertilize if in the bonus window and not already fertilized
          my_day=CROP_INFO[crop]["max_yield_day"]
          bonus_start=math.ceil(my_day/2)
          if(t.get("fertilized_until_day",-1)<day)and (bonus_start<=age<my_day):
            fertilize_jobs.append({"pos":(x,y),"action":["FERTILIZE"],"needs":FERTILIZER,"kind":"fertilize"})

        # Check if it's an animal structure (COOP/PASTURE)
        # Assuming structures have a 'type' key, e.g., t = {'type': 'COOP', ...}
        elif t.get("type") in ("COOP","PASTURE"):
          animal = t.get("animal")
          if animal is None:
            # This action seems to be for feeding, but no animal is present.
            # It might be intended for placing a new animal. Keeping as-is for now.
            place_jobs.append({"pos":(x,y),"action":["FEED"],"needs":WHEAT,"kind":"feed"})

          #Priority 2:Harvest Animal Products
          if t.get("yield_units",0)>0:
            harvest_jobs.append({"pos":(x,y),"action":["HARVEST"],"needs":None,"Kind":"harvest"})

          #Priority 3:Care for animal to bank yield bonuses
          if t.get("fed_today",False) and not t.get("cared_today",False):
            care_jobs.append({"pos":(x,y),"action":["CARE"],"needs":None,"kind":"care"})

          #Priority 4:Collect fertilizer
          if t.get("fertilizer_available",False):
            collect_fert_jobs.append({"pos":(x,y),"action":["COLLECT_FERTILIZER"],"needs":None,"kind":"collect_fert"})

  #Return tiers in strict priority order
  #Animals must not starve,plants must not weed out,ready produce should be banked.
  return[feed_jobs,water_jobs,harvest_jobs,care_jobs,collect_fert_jobs,fertilize_jobs,place_jobs,plant_jobs,weed_jobs]

In [19]:
from re import A
#5.WORKER ASSIGNMENT(The "Manager":Who does what?)
def assign_jobs(workers,job_tiers,shed_adjacent):
  "Tiered greedy nearest assignment."
  assignments=[None]*len(workers)
  unassigned=list(range(len(workers)))
  claimed_positions=set()

  for tier in job_tiers:
    if not unassigned or not tier:
      continue

    available=[j for j in tier if j["pos"] not in claimed_positions]
    still_unassigned=[]


    for i in unassigned:
      pos,_inv=workers[i]
      best,best_dist=None,None
      for j in available:
        if j["pos"] in claimed_positions:
          continue
        d = manhattan(pos,j["pos"])
        if best is None or d<best_dist:
          best,best_dist=j,d

      if best is None:
        assignments[i]=best

        claimed_positions.add(best["pos"])
      else:
        still_unassigned.append(i)

      unassigned = still_unassigned
    return assignments

def has_item(inv,item,qty=1):
  return inv.get(item,0)>=qty

def worker_actions_for_jobs(pos,inv,job,shed_adjacent,seeds,planned_crop):
  "Generates the exact action for worker based on their assigned job"
  if job is None:
    return["PASS"]

  target=job["pos"]
  needs=job["needs"]


  #Planting Logic
  if job["kind"]=="plant":
    crop = planned_crop
    if crop is None or not has_item(seeds,crop,1):
      return["PASS"]
    if pos == target:
      return["PLANT",crop]

    d = step_toward(pos,target)
    return[d] if d else ["PASS"]

  #Jobs needing a carried item(wheat/fertilizer/animal):fetch from shed first

  if needs in (WHEAT,FERTILIZER)and not has_item(inv,needs,1):
    if pos in shed_adjacent:
      return["PICKUP",needs,5]
    d = step_toward(pos,list(shed_adjacent)[0])
    return[d] if d else ["PASS"]

  if needs and needs.startswith("ANIMAL_FOR_"):
    structure = needs.replace("ANIMAL_FOR_","")
    animal=None

    for a,info in ANIMAL_INFO.items():
      if info["structure"]==structure and has_item(inv,a,1):
        animal= a
        break

      if animal is None:
        return["PASS"]
      if  pos == target:
        return["PLACE",animal,1]
      d=step_toward(pos,target)
      return[d] if d else ["PASS"]


  #Plain jobs:move to tile,then act

  if pos == target:
    return job["action"]

  d = step_toward(pos,target)

  return[d] if d else ["PASS"]


In [20]:
#6.Market & ECONOMY PLANNING(The "Economist")
def choose_planting_crop(money,day,shed,seeds):
  "Decides which crop to prioritize planting based on game day and cash."
  for crop in PLANT_PRIORITY:
    held = seeds.get(crop,0)
    cap= MAX_SEEDS_TO_HOLD.get(crop,0)
    if crop in (MELON,TOMATO,STRAWBERRY) and day<3:
      continue #Too expensive/slow for day 0-2

    if held<cap:
      return crop
  return WHEAT

def fib_hire_cost(n_already_hired):
  a,b = 1,1
  for _ in range(n_already_hired):
    a,b = b,a+b
  return a

def count_placed(farm,animal):
  n=0

  for row in farm["tiles"]:
    for t in row:
      if isinstance(t,dict) and t.get("animal") == animal:
        n+=1
  return n

def plan_market_orders(obs,farm,private,day,n_animals,n_coops,n_pastures,empty_tile_count,n_workers):
  money = farm["money"]
  shed = private["shed"]
  seeds = private["seeds"]
  orders=[]
  spend_planned=0


  def can_spend(amount,floor):
    return money - spend_planned-amount>= floor


#1)Sell everything sellable,keeping a wheat buffer for animal feed

  wheat_reserve = WHEAT_RESERVE_PER_ANIMAL * max(n_animals,1)
  for product in SELLABLE_PRODUCTS:
    have = shed.get(product,0)
    if product == WHEAT:
      sellable = max(0,have-wheat_reserve)
    else:
      sellable = have
    if sellable >0:
      orders.append(["SELL",product,sellable])

#2) Buy seeds to keep a small buffer of each crop we intend to plant
  for crop in PLANT_PRIORITY:
    if len(orders)>=10:break
    held = seeds.get(crop,0)
    cap = MAX_SEEDS_TO_HOLD.get(crop,0)
    cost = CROP_INFO[crop]["seed_cost"]
    if crop in (MELON,TOMATO,STRAWBERRY)  and day <3:
      continue

    while held<cap and can_spend(cost,CASH_BUFFER):
      orders.append(["BUY_SEED",crop,1]) # Corrected CROP to crop
      spend_planned +=cost
      held +=1
      if len(orders)>=10:break

#3)Buy fertilizer opportunistically once cash is comfortable
  if len(orders)<10 and shed.get(FERTILIZER,0)<3 and can_spend(100,600):
    orders.append(["BUY_PRODUCT",FERTILIZER,1])
    spend_planned +=100

#4)Buy animals if we have the structure and the cash
  if len(orders)<10:
    for animal,info in ANIMAL_INFO.items():
      have=shed.get(animal,0)
      structure_counts=n_coops if info["structure"] == "COOP" else n_pastures
      target = TARGET_ANIMALS[animal]
      if have + count_placed(farm,animal)<target and structure_counts>have and can_spend(info["cost"],ANIMAL_BUILD_MONEY[animal]):
        orders.append(["BUY_ANIMAL",animal,1])
        spend_planned += info["cost"]
        if len(orders)>=10:break


#5) Hire an extra hand if theres clearly more standing work than workers

  if len(orders)<10:
    hire_cost = fib_hire_cost(farm.get("hires_today",0))
    workload = max(0,25-empty_tile_count)# Rough proxy:more planted tiles=more upkeep work

    if workload>n_workers*4 and can_spend(hire_cost,HIRE_MONEY_FLOOR):
      orders.append(["HIRE"])
      spend_planned += hire_cost

#6) Buy the next land quadrant once the farm is getting crowded
  if len(orders)<10:
    unlocked=farm.get("unlocked_quadrants",[])
    for q in QUADRANT_ORDER:
      if q in unlocked:
        continue
      cost=LAND_COSTS[q]
      if empty_tile_count<=3 and can_spend(cost,LAND_BUY_FLOOR):
        orders.append(["BUY_LAND"]) # Corrected ORDERS.APPEND to orders.append
        spend_planned +=cost

      break

  return orders[:10]

def maybe_build_job(farm,money,n_coops,n_pastures):
  "If we want more animal capacity and can affoird it,return a build job."
  want_coop = n_coops<TARGET_ANIMALS[GOOSE] and money >= ANIMAL_BUILD_MONEY[GOOSE]
  want_pasture = (n_pastures<TARGET_ANIMALS[COW]+TARGET_ANIMALS[SHEEP] and money>=min(ANIMAL_BUILD_MONEY[COW],ANIMAL_BUILD_MONEY[SHEEP])) # Corrected n-pastures to n_pastures

  if not(want_coop or want_pasture):
    return None

  action = ["BUILD_COOP"] if want_coop else ["BUILD_PASTURE"]
  for y,row in enumerate(farm["tiles"]):
    for x,t in enumerate(row):
      if t is None:
        return{"pos":(x,y),"action":action,"needs":None,"kind":"build"}
  return None

In [21]:
!pip install kaggle-environments

In [22]:
#7.MAIN AGENT ENTRY POINT(The "Conductor")
def agent(obs):
  player=obs["player"]
  farm = obs["farms"][player]
  private=obs["private"]
  day=obs.get("day",0)

  tiles = farm["tiles"]
  board_size = len(tiles)
  half = board_size//2
  shed_adjacent = {(half-1,half-1),(half,half-1),(half-1,half),(half,half)}

  money=farm["money"]
  seeds=private["seeds"]
  shed=private["shed"]
  inventories = private["inventories"]

  farmer_pos = tuple(farm["farmer"])
  hand_positions = [tuple(p)for p in farm.get("hands",[])]
  all_positions= [farmer_pos] + hand_positions

  # Bundle workers with the current inventory
  workers=[(all_positions[i],inventories[i]if i<len(inventories)else{})for i in range(len(all_positions))]

  n_coops=sum(1 for row in tiles for t in row if isinstance(t,dict)and t.get("kind")=="COOP")
  n_pastures=sum(1 for row in tiles for t in row if isinstance(t,dict) and t.get("kind")=="PASTURE")
  n_animals= sum(1 for row in tiles for t in row if isinstance(t,dict) and t.get("animal")is not None)

  empty_tile_count=sum(1 for row in tiles for t in row if t is None)

  #--Step 1;Discover Jobs & Assign Workers--
  job_tiers = find_jobs(farm,day,shed_adjacent)
  build_job = maybe_build_job(farm,money,n_coops,n_pastures)

  #Insert build tier between "fertilize " and "place"
  job_tiers.insert(6,[build_job] if build_job else[])

  assignments = assign_jobs(workers,job_tiers,shed_adjacent)
  planned_crop = choose_planting_crop(money,day,shed,seeds)

  actions=[]
  for (pos,inv),job in zip(workers,assignments):
    actions.append(worker_actions_for_jobs(pos,inv,job,shed_adjacent,seeds,planned_crop))

  farmer_action=actions[0]if actions else["PASS"]
  hand_actions=actions[1:]if len(actions)>1 else[]

  #---Step 2:Plan Market Orders--
  market_orders=plan_market_orders(obs,farm,private,day,n_animals,n_coops,n_pastures,empty_tile_count,len(workers))


  return{"farmer":farmer_action,"hands":hand_actions,"market":market_orders}

#Local testing block
if __name__ == "__main__":
  from kaggle_environments import make
  import json

  print("Starting local test match with Adavanced Heuristic Agent..")
  env=make("kaggriculture",configuration={"episodeSteps":720},debug=True)
  env.run([agent,"random"])

  final_step=env.steps[-1]
  for i,state in enumerate(final_step):
    print(f"Player{i} Final Reward(Money):{state.reward},Status:{state.status}")

  with open("replay.json","w")as f:
    json.dump(env.toJSON(),f)
  print("Local test complete.Upload'replay.json' to Kaggle Visualizer to watch!")

Starting local test match with Adavanced Heuristic Agent..
Player0 Final Reward(Money):2319.0,Status:DONE
Player1 Final Reward(Money):0.0,Status:DONE
Local test complete.Upload'replay.json' to Kaggle Visualizer to watch!
